In [ ]:
import pandas as pd
from pathlib import Path
from counterfactual_generation.evaluate.metrics import (
    predict_with_sliding_window,
    compute_edit_distance,
    compute_perplexity,
)
from counterfactual_generation.evaluate.metrics import _majority_vote
from transformers import pipeline, AutoTokenizer
from counterfactual_generation.config import config

MODEL_MAP = {
    "imdb"  : "textattack/bert-base-uncased-imdb",
    "snli"  : "textattack/bert-base-uncased-snli",
    "agnews": "textattack/bert-base-uncased-ag-news",
}


def flip_rate_multiclass(orig_texts, cf_texts):
    clf_name = config.classifier_model
    clf      = pipeline("text-classification", model=clf_name, device=0)
    #tok      = AutoTokenizer.from_pretrained(clf_name, use_fast=False)
    tok = AutoTokenizer.from_pretrained(clf_name, use_fast=True)


    orig_preds = predict_with_sliding_window(orig_texts, clf, tok)
    cf_preds   = predict_with_sliding_window(cf_texts,  clf, tok)
    flips      = sum(o != c for o, c in zip(orig_preds, cf_preds))
    return flips / len(orig_texts), orig_preds, cf_preds


results_dir = Path("../results/results_nl")
csv_files   = sorted(results_dir.glob("*.csv"))
summary_rows = []

for file in csv_files:
    print(f"\n now evaluate: {file.name}")
    try:
        df = pd.read_csv(file)
        df.columns = [c.strip().lower() for c in df.columns]
        orig_col = next(c for c in df.columns if "original" in c)
        cf_col   = next(c for c in df.columns if "counterfactual" in c)

        originals = df[orig_col].astype(str).tolist()
        counterfs = df[cf_col].astype(str).tolist()

        dataset = next(ds for ds in MODEL_MAP if ds in file.stem)
        config.task_name = dataset
        config.classifier_model = MODEL_MAP[dataset]

        fr, _, _ = flip_rate_multiclass(originals, counterfs)
        ed = compute_edit_distance(originals, counterfs)
        ppl = compute_perplexity(counterfs)

        summary_rows.append({
            "file": file.name,
            "dataset": dataset,
            "flip_rate": round(fr, 3),
            "edit_distance": round(ed, 3),
            "perplexity": round(ppl, 2),
        })
        print(f"success | FR {fr:.3f} | ED {ed:.3f} | PPL {ppl:.2f}")

    except Exception as e:
        print(f"Error -> {file.name} (skipped)")
        print("    Error msg:", e)
        continue


        
pd.DataFrame(summary_rows)

2025-11-25 22:39:41.477809: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764081581.504101   21563 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764081581.511445   21563 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764081581.531763   21563 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764081581.531788   21563 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764081581.531791   21563 computation_placer.cc:177] computation placer alr


 now evaluate: results_nl_agnews_None_counterfactuals_20251001_002925.csv


Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Token indices sequence length is longer than the specified maximum sequence length for this model (891 > 512). Running this sequence through the model will result in indexing errors


  0%|          | 0/32 [00:00<?, ?it/s]

success | FR 0.729 | ED 5.363 | PPL 44.73

 now evaluate: results_nl_imdb_None_counterfactuals_20250930_195619.csv


...

success | FR 0.768 | ED 0.463 | PPL 43.81

 now evaluate: results_nl_snli_premise_counterfactuals_20250929_215137.csv


Device set to use cuda:0


  0%|          | 0/23 [00:00<?, ?it/s]

success | FR 0.638 | ED 0.354 | PPL 148.41


,file,dataset,flip_rate,edit_distance,perplexity
0,results_nl_agnews_None_counterfactuals_2025100...,agnews,0.729,5.363,44.73
1,results_nl_imdb_None_counterfactuals_20250930_...,imdb,0.984,0.926,36.47
2,results_nl_snli_hypothesis_counterfactuals_202...,snli,0.768,0.463,43.81
3,results_nl_snli_premise_counterfactuals_202509...,snli,0.638,0.354,148.41
